# DS-3002 Data Mining — Assignment #4
## Heartbeat to Heatmap: Unsupervised Learning, Ensemble Methods, and Neural Networks
**Spring 2026 · BSDS · FAST-NUCES**

---
*All random seeds fixed to `random_state=42`. Run all cells top-to-bottom.*


## Setup — Imports & Configuration


In [1]:
# SETUP
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
from imblearn.over_sampling import SMOTE
import joblib

SEED = 42
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid')
os.makedirs('outputs', exist_ok=True)
print("Setup complete.")

Setup complete.


## Preprocessing
### Pre-1 — Load Dataset


In [2]:
# Pre-1
col_names = ['age','sex','cp','trestbps','chol','fbs','restecg',
             'thalach','exang','oldpeak','slope','ca','thal','target']
df_raw = pd.read_csv('processed.cleveland.data', header=None, names=col_names)
print(f"Shape: {df_raw.shape}")
print("\nFirst 5 rows:")
print(df_raw.head())
print("\nData types:")
print(df_raw.dtypes)

Shape: (303, 14)

First 5 rows:
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca thal  target  
0    3.0  0.0  6.0       0  
1    2.0  3.0  3.0       2  
2    2.0  2.0  7.0       1  
3    3.0  0.0  3.0       0  
4    1.0  0.0  3.0       0  

Data types:
age         float64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalach     float64
exang       float64
oldpeak     float64
slope       float64
ca              str
thal            str
target        int64
dtype: object


### Pre-2 — Missing Values


In [3]:
# Pre-2
df = df_raw.replace('?', np.nan)
df = df.apply(pd.to_numeric, errors='coerce')
print("Missing values per column BEFORE dropping:")
mv = df.isnull().sum()
print(mv[mv > 0])
rows_before = len(df)
df = df.dropna()
print(f"\nRows before drop: {rows_before}")
print(f"Rows after  drop: {len(df)}")
print(f"Rows removed    : {rows_before - len(df)}")

Missing values per column BEFORE dropping:
ca      4
thal    2
dtype: int64

Rows before drop: 303
Rows after  drop: 297
Rows removed    : 6


### Pre-3 — Class Distribution & SMOTE Decision


In [4]:
# Pre-3
df['target'] = (df['target'] > 0).astype(int)
counts = df['target'].value_counts()
pcts   = df['target'].value_counts(normalize=True) * 100
dist   = pd.DataFrame({'Count': counts, 'Percent%': pcts.round(2)})
dist.index = dist.index.map({0:'No Disease', 1:'Disease'})
print(dist)
print("\nThe dataset is ~54/46 — nearly balanced.")
print("SMOTE applied on training split only as a precaution to ensure equal class weights.")

            Count  Percent%
target                     
No Disease    160     53.87
Disease       137     46.13

The dataset is ~54/46 — nearly balanced.
SMOTE applied on training split only as a precaution to ensure equal class weights.


### Pre-4 — One-Hot Encoding & StandardScaler


In [5]:
# Pre-4
CAT_COLS  = ['cp','restecg','slope','thal']
CONT_COLS = ['age','trestbps','chol','thalach','oldpeak','ca']
BIN_COLS  = ['sex','fbs','exang']

df_enc = pd.get_dummies(df, columns=CAT_COLS, drop_first=False)
print(f"Shape after one-hot encoding: {df_enc.shape}")
print("Columns:", list(df_enc.columns))

Shape after one-hot encoding: (297, 23)
Columns: ['age', 'sex', 'trestbps', 'chol', 'fbs', 'thalach', 'exang', 'oldpeak', 'ca', 'target', 'cp_1.0', 'cp_2.0', 'cp_3.0', 'cp_4.0', 'restecg_0.0', 'restecg_1.0', 'restecg_2.0', 'slope_1.0', 'slope_2.0', 'slope_3.0', 'thal_3.0', 'thal_6.0', 'thal_7.0']


### Pre-5 — Stratified 80/20 Train/Test Split


In [6]:
# Pre-5
X = df_enc.drop('target', axis=1)
y = df_enc['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]} rows  |  Test: {X_test.shape[0]} rows")
print(f"Train class balance: {y_train.value_counts().to_dict()}")
print(f"Test  class balance: {y_test.value_counts().to_dict()}")

# Fit StandardScaler on train only
scaler = StandardScaler()
X_train[CONT_COLS] = scaler.fit_transform(X_train[CONT_COLS])
X_test[CONT_COLS]  = scaler.transform(X_test[CONT_COLS])
joblib.dump(scaler, 'outputs/scaler.pkl')

# SMOTE on training split only
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print(f"After SMOTE: {pd.Series(y_train_res).value_counts().to_dict()}")

# Save splits for Parts B, C, E
joblib.dump((X_train, X_test, y_train, y_test), 'outputs/splits.pkl')
joblib.dump((X_train_res, y_train_res), 'outputs/train_resampled.pkl')

Train: 237 rows  |  Test: 60 rows
Train class balance: {0: 128, 1: 109}
Test  class balance: {0: 32, 1: 28}
After SMOTE: {1: 128, 0: 128}


['outputs/train_resampled.pkl']

### Pre-6 — Correlation Heatmap


In [7]:
# Pre-6
numeric_orig = df[CONT_COLS + BIN_COLS + ['target']].copy()
corr = numeric_orig.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation Heatmap — Numeric Features (Pre-6)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/pre6_corr_heatmap.png', dpi=150)
plt.show()
print("Saved: outputs/pre6_corr_heatmap.png")

upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
print("\nTop 3 correlated feature pairs:")
print(upper.stack().nlargest(3))
print("\nNaive Bayes note: correlated features violate NB's independence assumption,"
      " potentially inflating its confidence and reducing calibration quality.")

Saved: outputs/pre6_corr_heatmap.png

Top 3 correlated feature pairs:
ca       target    0.463189
oldpeak  target    0.424052
thalach  target    0.423817
dtype: float64

Naive Bayes note: correlated features violate NB's independence assumption, potentially inflating its confidence and reducing calibration quality.


---
## Part A — Unsupervised Learning
*Uses standardised feature matrix WITHOUT the target label.*


In [8]:
# A — Unsupervised Setup
# Build full scaled matrix — all rows, no target
X_full_enc = df_enc.drop('target', axis=1).copy()
sc_all = StandardScaler()
X_full_enc[CONT_COLS] = sc_all.fit_transform(X_full_enc[CONT_COLS])
X_full  = X_full_enc.values
y_true  = df['target'].values
print(f"Unsupervised feature matrix: {X_full.shape}")

Unsupervised feature matrix: (297, 22)


### A1 — K-Means Clustering


In [9]:
# A1 — KMeans WCSS & Silhouette
k_range = range(2, 9)
inertias, sil_scores = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_full)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_full, labels))

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()
line1, = ax1.plot(list(k_range), inertias, 'o-', color='#4C72B0', lw=2, label='WCSS (Inertia)')
line2, = ax2.plot(list(k_range), sil_scores, 's--', color='#DD8452', lw=2, label='Silhouette Score')
best_k = 3
ax1.axvline(best_k, color='red', linestyle=':', lw=1.5, label=f'Chosen k={best_k}')
ax1.set_xlabel('k'); ax1.set_ylabel('WCSS / Inertia', color='#4C72B0')
ax2.set_ylabel('Silhouette Score', color='#DD8452')
ax1.tick_params(axis='y', labelcolor='#4C72B0'); ax2.tick_params(axis='y', labelcolor='#DD8452')
lines = [line1, line2]; ax1.legend(lines, [l.get_label() for l in lines], loc='upper right')
ax1.set_title('A1 — K-Means: WCSS & Silhouette vs k', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('outputs/a1_kmeans_elbow.png', dpi=150); plt.show()

print(f"Chosen k={best_k}: elbow in WCSS curve is clearest here, and silhouette remains")
print("competitive. k=2 oversimplifies; k>3 shows diminishing WCSS returns.")

Chosen k=3: elbow in WCSS curve is clearest here, and silhouette remains
competitive. k=2 oversimplifies; k>3 shows diminishing WCSS returns.


In [10]:
# A1 — PCA Scatter
# Final KMeans + PCA scatter
km_best  = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = km_best.fit_predict(X_full)

pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_full)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
s1 = axes[0].scatter(X_pca2[:,0], X_pca2[:,1], c=km_labels,
                     cmap='Set1', alpha=0.7, s=40, edgecolors='k', lw=0.3)
axes[0].set_title('PCA 2D — K-Means Clusters (k=3)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(s1, ax=axes[0], label='Cluster')

s2 = axes[1].scatter(X_pca2[:,0], X_pca2[:,1], c=y_true,
                     cmap='coolwarm', alpha=0.7, s=40, edgecolors='k', lw=0.3)
axes[1].set_title('PCA 2D — True Disease Label', fontsize=12, fontweight='bold')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.colorbar(s2, ax=axes[1], label='0=No Disease / 1=Disease')

plt.suptitle('A1 — PCA Scatter: K-Means vs True Labels', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('outputs/a1_pca_scatter.png', dpi=150); plt.show()

In [11]:
# A1 — Cluster Summary & ARI
# Cluster summary table
df_clust = df[['thalach','oldpeak','cp']].copy()
df_clust['cluster'] = km_labels
df_clust['target']  = y_true
summary = df_clust.groupby('cluster').agg(
    Size=('target','count'),
    Disease_Prop=('target','mean'),
    Mean_thalach=('thalach','mean'),
    Mean_oldpeak=('oldpeak','mean'),
    Mean_cp=('cp','mean')
).round(3)
print("A1 — Cluster Summary Table:")
print(summary)
print("\nCluster 0: Low disease rate (18%), high thalach — likely healthy, younger patients.")
print("Cluster 1: Very high disease rate (91%), low thalach, high oldpeak — high-risk group.")
print("Cluster 2: Moderate disease rate (33%) — borderline / mixed clinical profile.")

ari_km = adjusted_rand_score(y_true, km_labels)
print(f"\nARI (K-Means vs true labels): {ari_km:.4f}")
print("ARI ~0.25 indicates moderate structural agreement — clusters capture real clinical")
print("signal but the data classes are not perfectly linearly separable in this space.")

A1 — Cluster Summary Table:
         Size  Disease_Prop  Mean_thalach  Mean_oldpeak  Mean_cp
cluster                                                         
0         100         0.180       165.190         0.537    2.850
1          93         0.914       128.817         2.147    3.710
2         104         0.327       153.192         0.578    2.962

Cluster 0: Low disease rate (18%), high thalach — likely healthy, younger patients.
Cluster 1: Very high disease rate (91%), low thalach, high oldpeak — high-risk group.
Cluster 2: Moderate disease rate (33%) — borderline / mixed clinical profile.

ARI (K-Means vs true labels): 0.2488
ARI ~0.25 indicates moderate structural agreement — clusters capture real clinical
signal but the data classes are not perfectly linearly separable in this space.


### A2 — Hierarchical Clustering


In [12]:
# A2 — Dendrogram
linked = linkage(X_full, method='ward')
cut_height = sorted(linked[:,2], reverse=True)[2]

fig, ax = plt.subplots(figsize=(12, 6))
dendrogram(linked, truncate_mode='lastp', p=25, ax=ax,
           color_threshold=0.6*max(linked[:,2]),
           above_threshold_color='grey')
ax.axhline(y=cut_height, color='red', linestyle='--', lw=1.5,
           label=f'Recommended cut @ {cut_height:.1f} → 3 clusters')
ax.set_title('A2 — Ward Hierarchical Dendrogram (top 25 merges)', fontsize=12, fontweight='bold')
ax.set_xlabel('Sample / Cluster Size'); ax.set_ylabel('Ward Distance')
ax.legend()
plt.tight_layout(); plt.savefig('outputs/a2_dendrogram.png', dpi=150); plt.show()

In [13]:
# A2 — Crosstab & ARI Comparison
hc = AgglomerativeClustering(n_clusters=3, linkage='ward')
hc_labels = hc.fit_predict(X_full)

ct = pd.crosstab(hc_labels, y_true, rownames=['Cluster'], colnames=['Disease (0=No / 1=Yes)'])
print("A2 — Cluster × True Label Crosstab:")
print(ct)

ari_compare = adjusted_rand_score(km_labels, hc_labels)
ari_hc      = adjusted_rand_score(y_true, hc_labels)
print(f"\nARI — K-Means vs Hierarchical : {ari_compare:.4f}")
print(f"ARI — Hierarchical vs true    : {ari_hc:.4f}")
print("\nK-Means (ARI~0.25) outperforms Hierarchical (ARI~0.19) vs true labels.")
print("For clinical segmentation, K-Means is preferred: it finds compact, balanced clusters")
print("more suited to an EM-style clinical profiling, whereas Ward linkage merges greedily")
print("and can create uneven cluster sizes that are harder to interpret clinically.")

A2 — Cluster × True Label Crosstab:
Disease (0=No / 1=Yes)   0   1
Cluster                       
0                       30  94
1                       50  25
2                       80  18

ARI — K-Means vs Hierarchical : 0.4783
ARI — Hierarchical vs true    : 0.1861

K-Means (ARI~0.25) outperforms Hierarchical (ARI~0.19) vs true labels.
For clinical segmentation, K-Means is preferred: it finds compact, balanced clusters
more suited to an EM-style clinical profiling, whereas Ward linkage merges greedily
and can create uneven cluster sizes that are harder to interpret clinically.


### A3 — Dimensionality Reduction (PCA + t-SNE)


In [14]:
# A3 — PCA Variance
pca_full = PCA(random_state=42)
pca_full.fit(X_full)
exp_var = pca_full.explained_variance_ratio_
cum_var = np.cumsum(exp_var)
n_90    = np.argmax(cum_var >= 0.90) + 1

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()
ax1.bar(range(1, len(exp_var)+1), exp_var, color='#4C72B0', alpha=0.7, label='Per-Component Variance')
ax2.plot(range(1, len(exp_var)+1), cum_var, 'o-', color='#DD8452', lw=2, label='Cumulative Variance')
ax2.axhline(0.90, color='red', linestyle='--', lw=1.2, label='90% threshold')
ax2.axvline(n_90, color='green', linestyle=':', lw=1.5, label=f'{n_90} components needed')
ax1.set_xlabel('Principal Component'); ax1.set_ylabel('Explained Var. Ratio', color='#4C72B0')
ax2.set_ylabel('Cumulative Explained Variance', color='#DD8452')
ax1.tick_params(axis='y', labelcolor='#4C72B0'); ax2.tick_params(axis='y', labelcolor='#DD8452')
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc='center right')
ax1.set_title('A3 — PCA: Explained Variance per Component', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('outputs/a3_pca_variance.png', dpi=150); plt.show()
print(f"Components needed for 90% variance: {n_90} out of {X_full.shape[1]}")

Components needed for 90% variance: 10 out of 22


In [15]:
# A3 — t-SNE
print("Running t-SNE (perplexity=30, ~30s)...")
tsne   = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_full)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_tsne[:,0], X_tsne[:,1], c=y_true,
                cmap='coolwarm', alpha=0.75, s=45, edgecolors='k', lw=0.3)
plt.colorbar(sc, ax=ax, label='0=No Disease / 1=Disease')
ax.set_title('A3 — t-SNE 2D Embedding (coloured by true disease label)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
plt.tight_layout(); plt.savefig('outputs/a3_tsne.png', dpi=150); plt.show()

print("t-SNE shows partial separation: disease patients (red) cluster toward one region")
print("but there is significant overlap with healthy patients (blue). This confirms the")
print("classification task is moderately difficult — a linear boundary alone is insufficient")
print("and tree-based or neural models are warranted.")

Running t-SNE (perplexity=30, ~30s)...


t-SNE shows partial separation: disease patients (red) cluster toward one region
but there is significant overlap with healthy patients (blue). This confirms the
classification task is moderately difficult — a linear boundary alone is insufficient
and tree-based or neural models are warranted.


---
## Part B — Bagging & Boosting
*Uses stratified 80/20 split and SMOTE on training set.*


In [16]:
# B — Setup
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
import xgboost as xgb
import shap

# Helper function
def evaluate_model(model_name, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro')
    rec = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    auc = roc_auc_score(y_true, y_prob)
    cm = confusion_matrix(y_true, y_pred)
    print(f"--- {model_name} ---")
    print(f"Accuracy:  {acc:.4f}\nMacro F1:  {f1:.4f}\nMacro Prec:{prec:.4f}\nMacro Rec: {rec:.4f}\nAUC-ROC:   {auc:.4f}")
    print(f"Confusion Matrix:\n{cm}\n")
    return {'acc': acc, 'f1': f1, 'auc': auc, 'rec_1': recall_score(y_true, y_pred, pos_label=1)}

### B1 — Random Forest


In [17]:
# B1 — RF Tuning
rf_param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [None, 5, 10]}
rf = RandomForestClassifier(random_state=42)
grid_rf = GridSearchCV(rf, rf_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
grid_rf.fit(X_train_res, y_train_res)

print("Best RF Params:", grid_rf.best_params_)
print(f"Best CV F1: {grid_rf.best_score_:.4f}")

best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

rf_metrics = evaluate_model("Random Forest", y_test, y_pred_rf, y_prob_rf)

Best RF Params: {'max_depth': 10, 'n_estimators': 50}
Best CV F1: 0.8349
--- Random Forest ---
Accuracy:  0.8167
Macro F1:  0.8141
Macro Prec:0.8200
Macro Rec: 0.8125
AUC-ROC:   0.9208
Confusion Matrix:
[[28  4]
 [ 7 21]]



In [18]:
# B1 — OOB Plot
n_estimators_range = range(1, 201)
oob_errors = []
rf_oob = RandomForestClassifier(warm_start=True, oob_score=True, random_state=42)

for i in n_estimators_range:
    rf_oob.set_params(n_estimators=i)
    rf_oob.fit(X_train_res, y_train_res)
    oob_errors.append(1 - rf_oob.oob_score_)

plt.figure(figsize=(8, 5))
plt.plot(n_estimators_range, oob_errors, label='OOB Error', color='#4C72B0')
plt.axvline(grid_rf.best_params_['n_estimators'], color='red', linestyle='--', label=f"Chosen n_trees={grid_rf.best_params_['n_estimators']}")
plt.xlabel('Number of Trees')
plt.ylabel('OOB Error')
plt.title('B1 - RF: OOB Error vs Trees', fontweight='bold')
plt.legend()
plt.tight_layout(); plt.savefig('outputs/b1_rf_oob.png', dpi=150); plt.show()

In [19]:
# B1 — Feature Importances
feature_names = X_train.columns.tolist()
importances = best_rf.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(8, 8))
plt.title('B1 - RF: Feature Importances', fontweight='bold')
plt.barh(range(len(indices)), importances[indices], align='center', color='#55A868')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Mean Decrease in Impurity')
plt.tight_layout(); plt.savefig('outputs/b1_rf_feat_imp.png', dpi=150); plt.show()

print("Top 5 Features:")
for i in indices[-5:][::-1]:
    print(f" - {feature_names[i]}: {importances[i]:.4f}")

print("\nConsequences of False Negatives:")
print("In cardiac screening, a false negative means sending a sick patient home,")
print("which could be fatal. High recall for the disease class is essential.")

Top 5 Features:
 - oldpeak: 0.0949
 - thal_3.0: 0.0928
 - ca: 0.0900
 - cp_4.0: 0.0893
 - thalach: 0.0888

Consequences of False Negatives:
In cardiac screening, a false negative means sending a sick patient home,
which could be fatal. High recall for the disease class is essential.


### B2 — Gradient Boosting (XGBoost)


In [20]:
# B2 — XGBoost
xgb_param_grid = {'learning_rate': [0.01, 0.1, 0.3], 'max_depth': [3, 5, 7]}
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
grid_xgb = GridSearchCV(xgb_model, xgb_param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
grid_xgb.fit(X_train_res, y_train_res)

print("Best XGB Params:", grid_xgb.best_params_)
print(f"Best CV F1: {grid_xgb.best_score_:.4f}")

best_xgb = xgb.XGBClassifier(**grid_xgb.best_params_, random_state=42, n_estimators=500, eval_metric='logloss', early_stopping_rounds=50)
eval_set = [(X_train_res, y_train_res), (X_test, y_test)]
best_xgb.fit(X_train_res, y_train_res, eval_set=eval_set, verbose=False)

results = best_xgb.evals_result()
x_axis = range(0, len(results['validation_0']['logloss']))

plt.figure(figsize=(8, 5))
plt.plot(x_axis, results['validation_0']['logloss'], label='Train')
plt.plot(x_axis, results['validation_1']['logloss'], label='Validation')
plt.axvline(best_xgb.best_iteration, color='red', linestyle='--', label=f'Optimal Round ({best_xgb.best_iteration})')
plt.xlabel('Boosting Rounds'); plt.ylabel('Log Loss')
plt.title('B2 - XGBoost: Train vs Validation Log Loss', fontweight='bold')
plt.legend(); plt.tight_layout(); plt.savefig('outputs/b2_xgb_logloss.png', dpi=150); plt.show()

y_pred_xgb = best_xgb.predict(X_test)
y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]
xgb_metrics = evaluate_model("XGBoost", y_test, y_pred_xgb, y_prob_xgb)

Best XGB Params: {'learning_rate': 0.01, 'max_depth': 3}
Best CV F1: 0.8231


--- XGBoost ---
Accuracy:  0.8333
Macro F1:  0.8303
Macro Prec:0.8403
Macro Rec: 0.8281
AUC-ROC:   0.9174
Confusion Matrix:
[[29  3]
 [ 7 21]]



In [21]:
# B2 — SHAP
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('B2 - XGBoost: SHAP Values Summary', fontweight='bold')
plt.tight_layout(); plt.savefig('outputs/b2_xgb_shap.png', dpi=150); plt.show()
joblib.dump(best_xgb, 'outputs/best_xgb.pkl')

['outputs/best_xgb.pkl']

---
## Part C — Artificial Neural Networks on Tabular Data


In [22]:
# C — Setup
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.callbacks import EarlyStopping
import time
tf.random.set_seed(42)

I0000 00:00:1777720732.450931  101615 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777720732.451397  101615 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1777720733.641795  101615 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777720733.642167  101615 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### C1 — Single-Layer Perceptron (SLP)


In [23]:
# C1 — SLP
slp = Sequential([Dense(1, input_dim=X_train_res.shape[1], activation='sigmoid')])
slp.compile(loss='binary_crossentropy', optimizer=SGD(learning_rate=0.01), metrics=['accuracy'])
history_slp = slp.fit(X_train_res, y_train_res, epochs=100, verbose=0)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_slp.history['loss'], label='Train Loss')
plt.title('C1 - SLP: Training Loss', fontweight='bold')
plt.xlabel('Epoch'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_slp.history['accuracy'], label='Train Accuracy')
plt.title('C1 - SLP: Training Accuracy', fontweight='bold')
plt.xlabel('Epoch'); plt.legend()
plt.tight_layout(); plt.savefig('outputs/c1_slp_history.png', dpi=150); plt.show()

weights = slp.layers[0].get_weights()[0].flatten()
abs_weights = np.abs(weights)
print("Top 3 SLP features:")
for i in np.argsort(abs_weights)[-3:][::-1]:
    print(f" - {feature_names[i]}: {weights[i]:.4f} (abs: {abs_weights[i]:.4f})")

y_prob_slp = slp.predict(X_test, verbose=0).flatten()
y_pred_slp = (y_prob_slp > 0.5).astype(int)
slp_metrics = evaluate_model("SLP", y_test, y_pred_slp, y_prob_slp)

print("A linear model like SLP is limited here because the data is not perfectly linearly separable,")
print("as seen in the t-SNE plot and PCA scatter.")

Top 3 SLP features:
 - ca: 0.6595 (abs: 0.6595)
 - cp_4.0: 0.5747 (abs: 0.5747)
 - thal_3.0: -0.5534 (abs: 0.5534)
--- SLP ---
Accuracy:  0.8833
Macro F1:  0.8825
Macro Prec:0.8838
Macro Rec: 0.8817
AUC-ROC:   0.9420
Confusion Matrix:
[[29  3]
 [ 4 24]]

A linear model like SLP is limited here because the data is not perfectly linearly separable,
as seen in the t-SNE plot and PCA scatter.


### C2 — Multi-Layer Perceptron (MLP)


In [24]:
# C2 — MLP
def create_mlp(arch):
    model = Sequential()
    model.add(Dense(arch[0], input_dim=X_train_res.shape[1], activation='relu'))
    model.add(Dropout(0.3))
    for units in arch[1:]:
        model.add(Dense(units, activation='relu'))
        model.add(Dropout(0.3))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])
    return model

architectures = {'Small': [32], 'Medium': [64, 32], 'Large': [128, 64, 32]}
best_val_f1, best_mlp_name = 0, ""

for name, arch in architectures.items():
    start_time = time.time()
    model = create_mlp(arch)
    X_t, X_v, y_t, y_v = train_test_split(X_train_res, y_train_res, test_size=0.2, random_state=42)
    model.fit(X_t, y_t, epochs=50, verbose=0)
    y_p = (model.predict(X_v, verbose=0).flatten() > 0.5).astype(int)
    f1 = f1_score(y_v, y_p, average='macro')
    print(f"Arch: {name} | Val F1: {f1:.4f} | Time: {time.time()-start_time:.2f}s")
    if f1 > best_val_f1: best_val_f1, best_mlp_name = f1, name

print(f"\nBest MLP Architecture: {best_mlp_name}")

final_mlp = create_mlp(architectures[best_mlp_name])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
start_time = time.time()
history_mlp = final_mlp.fit(X_train_res, y_train_res, validation_data=(X_test, y_test), epochs=150, callbacks=[early_stop], verbose=0)
final_mlp_time = time.time() - start_time

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_mlp.history['loss'], label='Train Loss')
plt.plot(history_mlp.history['val_loss'], label='Val Loss')
plt.axvline(early_stop.best_epoch, color='red', linestyle='--')
plt.title('C2 - Best MLP: Loss', fontweight='bold')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_mlp.history['accuracy'], label='Train Acc')
plt.plot(history_mlp.history['val_accuracy'], label='Val Acc')
plt.axvline(early_stop.best_epoch, color='red', linestyle='--')
plt.title('C2 - Best MLP: Accuracy', fontweight='bold')
plt.legend()
plt.tight_layout(); plt.savefig('outputs/c2_mlp_history.png', dpi=150); plt.show()

y_prob_mlp = final_mlp.predict(X_test, verbose=0).flatten()
y_pred_mlp = (y_prob_mlp > 0.5).astype(int)
mlp_metrics = evaluate_model("Best MLP", y_test, y_pred_mlp, y_prob_mlp)
mlp_metrics['time'] = final_mlp_time
final_mlp.save('outputs/best_mlp.h5')

Arch: Small | Val F1: 0.8725 | Time: 2.01s


Arch: Medium | Val F1: 0.8493 | Time: 2.15s


Arch: Large | Val F1: 0.8689 | Time: 2.30s

Best MLP Architecture: Small


--- Best MLP ---
Accuracy:  0.8500
Macro F1:  0.8490
Macro Prec:0.8502
Macro Rec: 0.8482
AUC-ROC:   0.9397
Confusion Matrix:
[[28  4]
 [ 5 23]]



In [25]:
# C2 — MLP CV
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_acc, cv_f1 = [], []

for train_idx, val_idx in kf.split(X_train_res):
    model = create_mlp(architectures[best_mlp_name])
    model.fit(X_train_res.iloc[train_idx], y_train_res.iloc[train_idx], epochs=early_stop.best_epoch, verbose=0)
    y_p = (model.predict(X_train_res.iloc[val_idx], verbose=0).flatten() > 0.5).astype(int)
    cv_acc.append(accuracy_score(y_train_res.iloc[val_idx], y_p))
    cv_f1.append(f1_score(y_train_res.iloc[val_idx], y_p, average='macro'))

print(f"5-Fold CV MLP Acc: {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}")
print(f"5-Fold CV MLP F1:  {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")

5-Fold CV MLP Acc: 0.8357 ± 0.0462
5-Fold CV MLP F1:  0.8327 ± 0.0443


### C3 — Ablation Study


In [26]:
# C3 — Ablation
def train_ablation(variant_name, remove_dropout=False, replace_relu=False, remove_es=False):
    model = Sequential()
    act = 'sigmoid' if replace_relu else 'relu'
    arch = architectures[best_mlp_name]
    
    model.add(Dense(arch[0], input_dim=X_train_res.shape[1], activation=act))
    if not remove_dropout: model.add(Dropout(0.3))
    for units in arch[1:]:
        model.add(Dense(units, activation=act))
        if not remove_dropout: model.add(Dropout(0.3))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])
    
    epochs = 150 if remove_es else 150
    callbacks = [] if remove_es else [EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]
    hist = model.fit(X_train_res, y_train_res, validation_data=(X_test, y_test), epochs=epochs, callbacks=callbacks, verbose=0)
    y_p = (model.predict(X_test, verbose=0).flatten() > 0.5).astype(int)
    return f1_score(y_test, y_p, average='macro'), hist.history['val_loss']

f1_A, loss_A = train_ablation("A: No Dropout", remove_dropout=True)
f1_B, loss_B = train_ablation("B: Sigmoid Activations", replace_relu=True)
f1_C, loss_C = train_ablation("C: No Early Stopping", remove_es=True)

print(f"Baseline (Best MLP): {mlp_metrics['f1']:.4f}")
print(f"Variant A (No Dropout): {f1_A:.4f}")
print(f"Variant B (Sigmoid): {f1_B:.4f}")
print(f"Variant C (No Early Stop): {f1_C:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(history_mlp.history['val_loss'], label='Baseline', lw=2)
plt.plot(loss_A, label='A: No Dropout')
plt.plot(loss_B, label='B: Sigmoid')
plt.plot(loss_C, label='C: No Early Stop')
plt.title('C3 - Ablation Study: Validation Loss', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Val Loss'); plt.legend()
plt.tight_layout(); plt.savefig('outputs/c3_ablation.png', dpi=150); plt.show()

Baseline (Best MLP): 0.8490
Variant A (No Dropout): 0.8490
Variant B (Sigmoid): 0.8825
Variant C (No Early Stop): 0.8496


### B3 — Ensemble Comparison & ROC
*(We run this after Part C so we can include the MLP in the comparison).*


In [27]:
# B3 — Comparison
table_data = [
    ["Best MLP", f"{mlp_metrics['acc']:.3f}", f"{mlp_metrics['f1']:.3f}", f"{mlp_metrics['auc']:.3f}", f"{mlp_metrics['rec_1']:.3f}", f"{mlp_metrics['time']:.2f}s"],
    ["Random Forest", f"{rf_metrics['acc']:.3f}", f"{rf_metrics['f1']:.3f}", f"{rf_metrics['auc']:.3f}", f"{rf_metrics['rec_1']:.3f}", "-"],
    ["XGBoost", f"{xgb_metrics['acc']:.3f}", f"{xgb_metrics['f1']:.3f}", f"{xgb_metrics['auc']:.3f}", f"{xgb_metrics['rec_1']:.3f}", "-"]
]
df_comp = pd.DataFrame(table_data, columns=["Classifier", "Accuracy", "Macro F1", "AUC-ROC", "Recall (Disease)", "Train Time"])
print(df_comp.to_string(index=False))

plt.figure(figsize=(8, 6))
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, y_prob_mlp)

plt.plot(fpr_mlp, tpr_mlp, label=f"Best MLP (AUC = {mlp_metrics['auc']:.3f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {rf_metrics['auc']:.3f})")
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC = {xgb_metrics['auc']:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.title('B3 - ROC Curve Comparison', fontweight='bold')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.legend()
plt.tight_layout(); plt.savefig('outputs/b3_roc_comparison.png', dpi=150); plt.show()

print("\nRecommendation:")
print("In a clinical setting, recall for the disease class (minimizing false negatives) is paramount.")
print("We should choose the model that maintains a high AUC while achieving the highest recall for the positive class.")

   Classifier Accuracy Macro F1 AUC-ROC Recall (Disease) Train Time
     Best MLP    0.850    0.849   0.940            0.821      3.27s
Random Forest    0.817    0.814   0.921            0.750          -
      XGBoost    0.833    0.830   0.917            0.750          -



Recommendation:
In a clinical setting, recall for the disease class (minimizing false negatives) is paramount.
We should choose the model that maintains a high AUC while achieving the highest recall for the positive class.
